> **Эта minibatch-ветка признана ненадёжной и больше не рекомендуется.** Для воспроизводимого grokking используйте `kaggle_sn_fullbatch_grokking.ipynb` и `train_sn_fullbatch.py`.

# Minibatch grokking на $S_5,S_6,S_7$

Архитектура следует **Stander et al., ICML 2024**: отдельные opaque-ID embeddings левого и правого операнда, конкатенация, один слой `Linear + ReLU` и unembedding.

В статье использовался full batch; этот notebook проверяет minibatch-вариант. Эксперимент с $S_7$ является новой экстраполяцией.

## Подключение кода

Загрузите `train_sn_minibatch.py` через **Add Input** как Dataset либо через панель Files. Следующая ячейка ищет файл в `/kaggle/input` и `/kaggle/working`.

In [ ]:
from pathlib import Path
import shutil
import sys

candidates = list(Path('/kaggle/input').rglob('train_sn_minibatch.py'))
local = Path('/kaggle/working/train_sn_minibatch.py')

if not local.exists():
    if not candidates:
        raise FileNotFoundError('Добавьте train_sn_minibatch.py как Kaggle Input')
    shutil.copy2(candidates[0], local)

print('Используется:', local)
sys.path.insert(0, '/kaggle/working')

## Конфигурация

Рекомендуется сначала оставить `n_values=(5,)`, затем отдельно запускать $S_6$ и $S_7$. Это экономит Kaggle-сессии и сразу показывает, сохраняется ли memorization gap при minibatch.


**Важно:** версия `v2_adamw` использует decoupled AdamW. В `v1` coupled `Adam(weight_decay=1)` в minibatch-режиме схлопывал веса к нулю. Старый checkpoint/`COMPLETED.json` не продолжаем: новое имя протокола создаёт чистый каталог.


In [ ]:
from train_sn_minibatch import Config, run

CONFIG = Config(
    output_root='/kaggle/working/sn_minibatch_grokking',
    protocol_name='stander_mlp_minibatch_v2_adamw',
    n_values=(5,),                 # затем (6,), затем (7,)
    seeds=(42,),
    train_fraction=0.40,
    batch_size_by_n={5: 4096, 6: 4096, 7: 2048},
    max_steps_by_n={5: 250_000, 6: 300_000, 7: 400_000},
    learning_rate=1e-3,
    weight_decay=1.0,
    required_gap_steps=10_000,
    resume_search_roots=('/kaggle/input',),
)
CONFIG

In [ ]:
RUN_DIRS = run(CONFIG)
RUN_DIRS

## Результаты

In [ ]:
import json

for run_dir in RUN_DIRS:
    result_path = run_dir / 'COMPLETED.json'
    if result_path.exists():
        print(run_dir)
        print(json.dumps(json.loads(result_path.read_text()), indent=2))

In [ ]:
import pandas as pd

for run_dir in RUN_DIRS:
    log_path = run_dir / 'training_log.csv'
    if log_path.exists():
        frame = pd.read_csv(log_path)
        display(frame.tail())

## Продолжение в новой Kaggle-сессии

После завершения сессии нажмите **Save Version**. В следующем notebook добавьте output прошлой версии через **Add Input → Your Work** и запустите те же ячейки. Скрипт автоматически импортирует соответствующий checkpoint из `/kaggle/input`.